In [1]:
from frappy.client import SecopClient

url = "localhost:10801"


gas_dosing: SecopClient = SecopClient(url)

gas_dosing.connect()

sim_gas_dosing ready


In [2]:
gas_dosing.descriptive_data

{'modules': {'massflow_contr1': {'accessibles': {'value': {'description': 'Mass flow of gas',
     'datainfo': {'unit': 'mln/min',
      'min': 0.0,
      'max': 1000.0,
      'type': 'double'},
     'readonly': True},
    'status': {'description': 'current status of the module',
     'datainfo': {'type': 'tuple',
      'members': [{'type': 'enum',
        'members': {'IDLE': 100, 'WARN': 200, 'BUSY': 300, 'ERROR': 400}},
       {'type': 'string'}]},
     'readonly': True},
    'target': {'description': 'Desired mass flow of gas',
     'datainfo': {'unit': 'mln/min',
      'min': 0.0,
      'max': 200.0,
      'type': 'double'},
     'readonly': False},
    'pollinterval': {'description': 'polling interval',
     'datainfo': {'min': 0.0, 'type': 'double'},
     'readonly': False},
    'ramp': {'description': 'desired ramp speed for gas flow of gas',
     'datainfo': {'unit': 'mln/min^2',
      'min': 0.0,
      'max': 200.0,
      'type': 'double'},
     'readonly': False},
    'setpoi

In [3]:
def extract_modules_and_parameters(data):
    modules = data.get("modules", {})
    module_list = list(modules.keys())

    parameter_map = {}

    for module_name, module_info in modules.items():
        params = module_info.get("accessibles", {})
        parameter_list = []

        for param_name, param_info in params.items():
            datainfo = param_info.get("datainfo", {})

        
            parameter_list.append(param_name)

        parameter_map[module_name] = parameter_list

    return module_list, parameter_map

In [ ]:
modulelist , para_List = extract_modules_and_parameters(gas_dosing.descriptive_data)

try to reconnect to localhost:10801
can not connect to localhost:10801, [Errno 61] Connection refused
can not reconnect to sim_gas_dosing (CommunicationFailed('can not connect to localhost:10801, [Errno 61] Connection refused'))
continue trying to reconnect
sim_gas_dosing ready


In [5]:
para_List[modulelist[0]],modulelist

(['value',
  'status',
  'target',
  'pollinterval',
  'ramp',
  'setpoint',
  'stop',
  '_factory_reset',
  '_inverse_calibration_coefficients',
  '_forward_calibration_coefficients',
  '_value_uncalibrated',
  '_target_calibrated',
  'target_limits',
  '_gastype',
  '_tolerance',
  '_P_value',
  '_I_value',
  '_D_value',
  '_gastype_select',
  '_valveposition',
  '_control_mode',
  '_test_cmd'],
 ['massflow_contr1',
  'massflow_contr2',
  'massflow_contr3',
  'massflow_contr4',
  'massflow_contr5',
  'massflow_contr6',
  'massflow_contr7',
  'massflow_contr8',
  'massflow_contr9',
  'massflow_contr10',
  'backpress_contr1',
  'backpress_contr2',
  'valve_controller',
  'gasflow_calibrator'])

In [12]:
from spin.config import Config, Device, Page, Param
params = []
for para_name in para_List[modulelist[0]]:
    params.append(Param(para_name))
params    

[Param(name='value', pollinterval=None, poll_every=1),
 Param(name='status', pollinterval=None, poll_every=1),
 Param(name='target', pollinterval=None, poll_every=1),
 Param(name='pollinterval', pollinterval=None, poll_every=1),
 Param(name='ramp', pollinterval=None, poll_every=1),
 Param(name='setpoint', pollinterval=None, poll_every=1),
 Param(name='stop', pollinterval=None, poll_every=1),
 Param(name='_factory_reset', pollinterval=None, poll_every=1),
 Param(name='_inverse_calibration_coefficients', pollinterval=None, poll_every=1),
 Param(name='_forward_calibration_coefficients', pollinterval=None, poll_every=1),
 Param(name='_value_uncalibrated', pollinterval=None, poll_every=1),
 Param(name='_target_calibrated', pollinterval=None, poll_every=1),
 Param(name='target_limits', pollinterval=None, poll_every=1),
 Param(name='_gastype', pollinterval=None, poll_every=1),
 Param(name='_tolerance', pollinterval=None, poll_every=1),
 Param(name='_P_value', pollinterval=None, poll_every=1),

In [11]:
import asyncio
import socket
import json

class DiscoveryProtocol(asyncio.DatagramProtocol):
    def __init__(self):
        self.transport = None
        self.responses = []

    def connection_made(self, transport):
        self.transport = transport

    def datagram_received(self, data, addr):
        try:
            response = data.decode('utf-8').strip()
            # Try to parse as JSON
            try:
                desc = json.loads(response)
                port = desc.get('port', addr[1])
                equipment_id = desc.get('equipment_id', None) or desc.get('name', None) or ''
                node_info = {
                    'host': addr[0],
                    'port': port,
                    'equipment_id': equipment_id,
                    'description': desc
                }
            except Exception as e:
                # Fallback: not JSON, just store raw response
                node_info = {
                    'host': addr[0],
                    'port': addr[1],
                    'equipment_id': '',
                    'description': response
                }
            if not any(n['host'] == node_info['host'] and n['port'] == node_info['port'] for n in self.responses):
                self.responses.append(node_info)
        except Exception as e:
            print(f"Error parsing discovery response: {e}")

async def discover_secop_nodes(timeout=2.0):
    DISCOVERY_PORT = 10767
    loop = asyncio.get_event_loop()
    protocol = DiscoveryProtocol()
    try:
        transport, _ = await loop.create_datagram_endpoint(
            lambda: protocol,
            local_addr=('0.0.0.0', 0),
            family=socket.AF_INET,
            allow_broadcast=True
        )
        discovery_msg = json.dumps({'SECoP': 'discover'}).encode('utf-8')
   
        transport.sendto(discovery_msg, ('255.255.255.255', DISCOVERY_PORT))
        await asyncio.sleep(timeout)
        discovered_nodes = protocol.responses
        print(discovered_nodes)
        transport.close()
    except Exception as e:
        print(f"Discovery error: {e}")
        discovered_nodes = []
    return discovered_nodes


In [12]:
discover_secop_nodes()

<coroutine object discover_secop_nodes at 0x773a5817a440>

In [13]:
# Test UDP SECoP discovery function in Jupyter
import nest_asyncio


nodes = await discover_secop_nodes(timeout=2.0)
nodes

[{'host': '134.30.134.134', 'port': 10805, 'equipment_id': 'ROCK_IT_abstraction_layer', 'description': {'SECoP': 'node', 'port': 10805, 'equipment_id': 'ROCK_IT_abstraction_layer', 'firmware': 'FRAPPY 0.20.4', 'description': 'SEC node for abstracted control methods used in the project ROCK-IT at HZB'}}, {'host': '134.30.134.134', 'port': 10804, 'equipment_id': 'gas_analysis_myspot_sim', 'description': {'SECoP': 'node', 'port': 10804, 'equipment_id': 'gas_analysis_myspot_sim', 'firmware': 'FRAPPY 0.20.4', 'description': 'Gas Analysis\n\nCollection of functionalities needed for the analysis of the gas after and before the catalytic process.'}}, {'host': '134.30.134.134', 'port': 10814, 'equipment_id': 'Hiden_MS', 'description': {'SECoP': 'node', 'port': 10814, 'equipment_id': 'Hiden_MS', 'firmware': 'FRAPPY 0.20.4', 'description': 'Hiden_MS\n\nCollection of functionalities needed for the analysis of the gas after and before the catalytic process.'}}, {'host': '134.30.134.134', 'port': 10

[{'host': '134.30.134.134',
  'port': 10805,
  'equipment_id': 'ROCK_IT_abstraction_layer',
  'description': {'SECoP': 'node',
   'port': 10805,
   'equipment_id': 'ROCK_IT_abstraction_layer',
   'firmware': 'FRAPPY 0.20.4',
   'description': 'SEC node for abstracted control methods used in the project ROCK-IT at HZB'}},
 {'host': '134.30.134.134',
  'port': 10804,
  'equipment_id': 'gas_analysis_myspot_sim',
  'description': {'SECoP': 'node',
   'port': 10804,
   'equipment_id': 'gas_analysis_myspot_sim',
   'firmware': 'FRAPPY 0.20.4',
   'description': 'Gas Analysis\n\nCollection of functionalities needed for the analysis of the gas after and before the catalytic process.'}},
 {'host': '134.30.134.134',
  'port': 10814,
  'equipment_id': 'Hiden_MS',
  'description': {'SECoP': 'node',
   'port': 10814,
   'equipment_id': 'Hiden_MS',
   'firmware': 'FRAPPY 0.20.4',
   'description': 'Hiden_MS\n\nCollection of functionalities needed for the analysis of the gas after and before the cata

In [4]:
# For Jupyter: run async code in a cell using nest_asyncio and await
import nest_asyncio
nest_asyncio.apply()

nodes = await discover_secop_nodes(timeout=2.0)
nodes

[]

In [29]:
# UDP SECoP node discovery using frappy protocol (JSON-based)
import asyncio
import socket
import json

class FrappyDiscoveryProtocol(asyncio.DatagramProtocol):
    def __init__(self):
        self.transport = None
        self.responses = []

    def connection_made(self, transport):
        self.transport = transport

    def datagram_received(self, data, addr):
        try:
            response = data.decode('utf-8').strip()
            desc = json.loads(response)
            if desc.get('SECoP') == 'node':
                node_info = {
                    'host': addr[0],
                    'port': desc.get('port', addr[1]),
                    'equipment_id': desc.get('equipment_id', ''),
                    'firmware': desc.get('firmware', ''),
                    'description': desc.get('description', ''),
                }
                if not any(n['host'] == node_info['host'] and n['port'] == node_info['port'] for n in self.responses):
                    self.responses.append(node_info)
        except Exception as e:
            print(f"Error parsing discovery response: {e}")

async def discover_frappy_secop_nodes(timeout=2.0):
    DISCOVERY_PORT = 10767
    loop = asyncio.get_event_loop()
    protocol = FrappyDiscoveryProtocol()
    try:
        transport, _ = await loop.create_datagram_endpoint(
            lambda: protocol,
            local_addr=('0.0.0.0', 0),
            family=socket.AF_INET,
            allow_broadcast=True
        )
        discovery_msg = json.dumps({'SECoP': 'discover'}).encode('utf-8')
        transport.sendto(discovery_msg, ('255.255.255.255', DISCOVERY_PORT))
        await asyncio.sleep(timeout)
        discovered_nodes = protocol.responses
        transport.close()
    except Exception as e:
        print(f"Discovery error: {e}")
        discovered_nodes = []
    return discovered_nodes

# Example usage in Jupyter:
import nest_asyncio
nest_asyncio.apply()
nodes = await discover_frappy_secop_nodes(timeout=2.0)
for node in nodes:
    print(node)
nodes

{'host': '134.30.59.80', 'port': 10802, 'equipment_id': 'sim_reactor_cell', 'firmware': 'FRAPPY 0.20.4', 'description': 'Reactor Cell\n\nCollection of functionalities needed for the temperature at the sample position and for the readout of pressures directly before and after the reactor cell.'}
{'host': '134.30.59.80', 'port': 10801, 'equipment_id': 'sim_gas_dosing', 'firmware': 'FRAPPY 0.20.4', 'description': 'Gas dosing\n\nGas dosing Node, with simulated hardware'}


[{'host': '134.30.59.80',
  'port': 10802,
  'equipment_id': 'sim_reactor_cell',
  'firmware': 'FRAPPY 0.20.4',
  'description': 'Reactor Cell\n\nCollection of functionalities needed for the temperature at the sample position and for the readout of pressures directly before and after the reactor cell.'},
 {'host': '134.30.59.80',
  'port': 10801,
  'equipment_id': 'sim_gas_dosing',
  'firmware': 'FRAPPY 0.20.4',
  'description': 'Gas dosing\n\nGas dosing Node, with simulated hardware'}]